# Crude Oil Price & Demand Forecasting Pipeline (Updated: XGBoost Demand)  
**Author**: Rayya Roy  
**Date**: 2025-04-28**  

**Objective:** Forecast monthly WTI price and OECD vs. non-OECD demand with enhanced XGBoost models.  

## 1. Setup & Environment

In [ ]:
# Install & import packages
# Ensure required packages are installed
%pip install requests pandas python-dotenv fredapi numpy matplotlib seaborn statsmodels==0.13.5 prophet scikit-learn xgboost seaborn --force-reinstall
%pip install pandas
import sys
sys.path.append('/usr/local/lib/python3.9/site-packages')  # Adjust the path if necessary

# Import necessary libraries
import os
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
from fredapi import Fred
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load API keys
load_dotenv()
EIA_API_KEY  = os.getenv("EIA_API_KEY")
FRED_API_KEY = os.getenv("FRED_API_KEY")
assert EIA_API_KEY,  "EIA_API_KEY not set"
assert FRED_API_KEY, "FRED_API_KEY not set"

## 2. Data Ingestion

In [ ]:
# 2.1 EIA WTI price
eia_url = f"https://api.eia.gov/series/?api_key={EIA_API_KEY}&series_id=PET.RWTC.M"
wti_json = requests.get(eia_url).json()
wti = pd.DataFrame(wti_json["series"][0]["data"], columns=["Date","WTI"])
wti["Date"] = pd.to_datetime(wti["Date"])
wti.set_index("Date", inplace=True)

# 2.2 BP demand
bp = pd.read_excel("bp_stat_review.xlsx", sheet_name="Consumption")
bp = bp.rename(columns={"Year":"Date"}).assign(Date=lambda df: pd.to_datetime(df["Date"].astype(str)))
bp.set_index("Date", inplace=True)
demand = bp[["OECD","Non-OECD"]].resample("MS").ffill()

# 2.3 FRED macro
fred = Fred(api_key=FRED_API_KEY)
gdp    = fred.get_series("GDP").resample("MS").ffill()
cpi    = fred.get_series("CPIAUCSL").resample("MS").ffill()
indpro = fred.get_series("INDPRO").resample("MS").ffill()

# 2.4 Geopolitical events
events = pd.read_csv("geopolitical_events.csv", parse_dates=["date"])

## 3. Preprocessing

In [ ]:
# Merge & resample
df = pd.concat([
    wti,
    demand,
    gdp.rename("GDP"),
    cpi.rename("CPI"),
    indpro.rename("INDPRO")
], axis=1).resample("MS").mean()

# Interpolate
df.interpolate(method="time", inplace=True)

# Standardize
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df),
    index=df.index,
    columns=df.columns
)

# Event flags
for evt in events["event"].unique():
    dates = events.loc[events.event==evt, "date"]
    df_scaled[evt] = df_scaled.index.isin(dates).astype(int)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12,10))
sns.heatmap(df_scaled.corr(), annot=True, fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.savefig("forecast_plots/corr_heatmap.png")
plt.show()

# Seasonal decomposition of WTI
res = seasonal_decompose(df["WTI"], model="additive", period=12)
res.plot()
plt.savefig("forecast_plots/wti_seasonal_decomp.png")
plt.show()

## 5. Feature Engineering

In [ ]:
lags = [1,3,6,12]
for lag in lags:
    df_scaled[f"WTI_lag{lag}"] = df_scaled["WTI"].shift(lag)
    df_scaled[f"OECD_lag{lag}"] = df_scaled["OECD"].shift(lag)
    df_scaled[f"Non-OECD_lag{lag}"] = df_scaled["Non-OECD"].shift(lag)

df_scaled["WTI_roll3"]  = df_scaled["WTI"].rolling(3).mean()
df_scaled["WTI_roll12"] = df_scaled["WTI"].rolling(12).mean()

df_fe = df_scaled.dropna()

## 6. Modeling

In [ ]:
# Train/test split
split_date = "2020-01-01"
train = df_fe.loc[:split_date]
test  = df_fe.loc[ split_date:]

X_train = train.drop(["WTI","OECD","Non-OECD"], axis=1)
y_train_price = train["WTI"]
y_train_oecd  = train["OECD"]
y_train_noec  = train["Non-OECD"]

X_test  = test.drop(["WTI","OECD","Non-OECD"], axis=1)
y_test_price = test["WTI"]
y_test_oecd  = test["OECD"]
y_test_noec  = test["Non-OECD"]

# SARIMA for price
sarima_exog = SARIMAX(
    train["WTI"],
    exog=train.drop(["WTI","OECD","Non-OECD"], axis=1),
    order=(1,1,1), seasonal_order=(1,1,1,12)
).fit(disp=False)

# Prophet for price
prophet_df = train[["WTI"]].reset_index().rename(columns={"Date":"ds","WTI":"y"})
m = Prophet()
m.add_regressor("CPI")
m.fit(prophet_df.merge(df_scaled[["CPI"]].reset_index(), on="ds"))
future = m.make_future_dataframe(periods=len(test), freq="MS")
future = future.merge(df_scaled[["CPI"]].reset_index(), on="ds", how="left")
prophet_forecast = m.predict(future)

# Ridge for demand
ridge_oecd = Ridge().fit(X_train, y_train_oecd)
ridge_noec = Ridge().fit(X_train, y_train_noec)

# XGBoost for price
xgb_price = XGBRegressor(n_estimators=500, max_depth=4).fit(X_train, y_train_price)

# XGBoost for demand
xgb_oecd = XGBRegressor(n_estimators=500, max_depth=4).fit(X_train, y_train_oecd)
xgb_noec = XGBRegressor(n_estimators=500, max_depth=4).fit(X_train, y_train_noec)

## 7. Validation & Metrics

In [ ]:
def eval_model(true, pred):
    rmse = np.sqrt(mean_squared_error(true, pred))
    mae  = mean_absolute_error(true, pred)
    mape = np.mean(np.abs((true - pred) / true)) * 100
    return rmse, mae, mape

metrics = []

# Price models
sarima_pred = sarima_exog.predict(start=test.index[0], end=test.index[-1], exog=X_test)
metrics.append(("SARIMA",) + eval_model(y_test_price, sarima_pred))

prop_pred = prophet_forecast.set_index("ds")["yhat"].loc[test.index]
metrics.append(("Prophet",) + eval_model(y_test_price, prop_pred))

xgb_price_pred = xgb_price.predict(X_test)
metrics.append(("XGB_Price",) + eval_model(y_test_price, xgb_price_pred))

# Demand models
ridge_oecd_pred = ridge_oecd.predict(X_test)
metrics.append(("Ridge_OECD",) + eval_model(y_test_oecd, ridge_oecd_pred))

xgb_oecd_pred = xgb_oecd.predict(X_test)
metrics.append(("XGB_OECD",) + eval_model(y_test_oecd, xgb_oecd_pred))

ridge_noec_pred = ridge_noec.predict(X_test)
metrics.append(("Ridge_nonOECD",) + eval_model(y_test_noec, ridge_noec_pred))

xgb_noec_pred = xgb_noec.predict(X_test)
metrics.append(("XGB_nonOECD",) + eval_model(y_test_noec, xgb_noec_pred))

# Compile metrics
metrics_df = pd.DataFrame(metrics, columns=["Model","RMSE","MAE","MAPE"])
metrics_df.to_csv("metrics_summary.csv", index=False)
metrics_df

## 8. Visualization of Forecasts

In [ ]:
# Price Forecast Plot
plt.figure(figsize=(10,5))
plt.plot(train["WTI"], label="Train")
plt.plot(test["WTI"], label="Test")
plt.plot(sarima_pred, label="SARIMA")
plt.plot(xgb_price_pred, label="XGB_Price")
plt.legend()
plt.title("WTI Price: Actual vs Forecasts")
plt.savefig("forecast_plots/wti_price_forecasts.png")
plt.show()

# Demand Feature Importance
imp_oecd = pd.Series(xgb_oecd.feature_importances_, index=X_train.columns).sort_values(ascending=False)[:10]
imp_non = pd.Series(xgb_noec.feature_importances_, index=X_train.columns).sort_values(ascending=False)[:10]
fig, axes = plt.subplots(1,2, figsize=(14,5))
imp_oecd.plot.bar(ax=axes[0], title="Top 10 Features: OECD Demand")
imp_non.plot.bar(ax=axes[1], title="Top 10 Features: non-OECD Demand")
plt.tight_layout()
plt.savefig("forecast_plots/demand_feature_importance.png")
plt.show()

## 9. Save Models

In [ ]:
import pickle
os.makedirs("models", exist_ok=True)
with open("models/sarima_results.pkl", "wb") as f: pickle.dump(sarima_exog, f)
with open("models/xgb_price.pkl", "wb") as f: pickle.dump(xgb_price, f)
with open("models/ridge_oecd.pkl", "wb") as f: pickle.dump(ridge_oecd, f)
with open("models/xgb_oecd.pkl", "wb") as f: pickle.dump(xgb_oecd, f)
with open("models/ridge_noec.pkl", "wb") as f: pickle.dump(ridge_noec, f)
with open("models/xgb_noec.pkl", "wb") as f: pickle.dump(xgb_noec, f)

## 10. Reports

- Executive_Summary.md (generated separately)
- Technical_Report.md (generated separately)